In [ ]:
from utils.api_client import GBFSClient
from config import GBFS_FEED
import pandas as pd

client = GBFSClient(GBFS_FEED)
client.get_data_feeds()

In [ ]:
station_feed = client.fetch_data("station_information")
# station_feed["last_updated"]

station_info = station_feed["data"]["stations"]
station_info[0]

In [ ]:
info_df = pd.DataFrame(station_info)
info_df.info()

In [ ]:
#replace missing config, rental methods, and postal code with "UNKNOWN"
info_df.columns[info_df.isnull().any()].tolist()

In [ ]:
info_df[info_df["rental_methods"].isna()]

In [ ]:
info_df["rental_methods"][:10]

In [ ]:
info_df["post_code"][:10]

In [ ]:
print(info_df["is_valet_station"].unique())
info_df[info_df["is_valet_station"].eq(True)].shape[0]

In [ ]:
# depreciated schema design
# info_df["_valet_station_details"].unique()
# info_df[(info_df["is_valet_station"] == True) & (info_df["_valet_station_details"].notna())].shape[0]

In [ ]:
cleaned_info = info_df.drop(columns=["vehicle_types_capacity", "vehicle_docks_capacity", "rental_uris", "rental_methods", "cross_street", "post_code"])

cleaned_info["name"] = cleaned_info["name"].apply(lambda x: GBFSClient.map_data(x, "language", "text").get("en"))
cleaned_info["short_name"] = cleaned_info["short_name"].apply(lambda x: GBFSClient.map_data(x, "language", "text").get("en"))
cleaned_info[info_df["address"].isna()]


In [ ]:
cleaned_info[cleaned_info["name"] == cleaned_info["address"]].shape[0]

In [ ]:
cleaned_info["address"] = cleaned_info["address"].fillna(cleaned_info["name"])
cleaned_info["is_valet_station"] = cleaned_info["is_valet_station"].astype(bool).fillna(False)
# cleaned_info["_valet_station_details"] = cleaned_info["_valet_station_details"].fillna("N/A")
cleaned_info.info()

In [ ]:
import geopandas as gpd
import folium

gdf = gpd.GeoDataFrame(
    cleaned_info,
    geometry=gpd.points_from_xy(cleaned_info.lon, cleaned_info.lat),
    crs="EPSG:4326"
)

map = folium.Map(location=[43.67, -79.38], zoom_start=13)

folium.GeoJson(
    gdf,
    name="Stations",
    popup=folium.GeoJsonPopup(
        fields=["name", "address", "capacity"],
        aliases=["Station Name:", "Address:", "Capacity:"]
    ),
    tooltip=folium.GeoJsonTooltip(fields=["name"], aliases=[""])
).add_to(map)

map

In [ ]:
status_feed = client.fetch_data("station_status")
status_data = status_feed["data"]["stations"]
status_data[0]

In [ ]:
from collections import Counter

num_dock_type_list = Counter(len(station["vehicle_docks_available"]) for station in status_data)
print(f"Num dock type lists: {num_dock_type_list}\n")

# since there is consistantly only 1 type list, check consistency of schema
num_dockable_types = Counter(len(station["vehicle_docks_available"][0]["vehicle_type_ids"]) for station in status_data)
print(f"Num types dockable: {num_dockable_types}")

type_id_list = status_data[0]["vehicle_docks_available"][0]["vehicle_type_ids"]
num_same_dockable = Counter(station["vehicle_docks_available"][0]["vehicle_type_ids"] == type_id_list for station in status_data)
print(f"Same types dockable: {num_same_dockable}\n")

num_types_avail = Counter(len(station["vehicle_types_available"]) for station in status_data)
print(f"Num types available: {num_types_avail}")

# check availability schema consistency
type_id_avail = GBFSClient.map_data(status_data[0]["vehicle_types_available"], "vehicle_type_id", "count").keys()
same_types_avail = Counter(GBFSClient.map_data(station["vehicle_types_available"], "vehicle_type_id", "count").keys() == type_id_avail for station in status_data)
print(f"Same types available: {same_types_avail}")

# last check, if dockable and available contain the same types
Counter(type_id_list) == Counter(type_id_avail)

In [ ]:
status_df = pd.DataFrame(status_data)
status_df.info()

In [ ]:
# this means vehicle_docks_available is droppable (count maintained in num_docks_available)
# vehicle_types_available will be flattened since vehicle types remain consistent
cleaned_status = status_df.drop(columns=["vehicle_docks_available", "vehicle_types_available"])
cleaned_status = cleaned_status.join(status_df["vehicle_types_available"].apply(lambda x: GBFSClient.map_data(x, "vehicle_type_id", "count")).apply(pd.Series).add_prefix("available_"))
cleaned_status.head()

In [ ]:
# confirm all records saved in utc
print(cleaned_status["last_reported"].str.endswith("Z").value_counts())

# check millisecond precision
print(cleaned_status["last_reported"].str.extract(r'\.(\d+)Z$')[0].str.len().value_counts())

In [ ]:
# not all timestamps have millisecond precision, use ISO8601
cleaned_status["last_reported"] = pd.to_datetime(cleaned_status["last_reported"], format="ISO8601", utc=True).dt.tz_convert(None).dt.floor("ms")
cleaned_status = cleaned_status.rename(columns=str.lower)
cleaned_status.head()

In [ ]:
cleaned_status.info()